### Adagrad - optimize learning rate

In [1]:
# implemented by HuyIGW04
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms as tf    # for convert tensor
from torch.utils.data import DataLoader

In [2]:
# setup device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [3]:
train_set = datasets.FashionMNIST(root='FashionMNIST-data',
                                  train=True,
                                  transform=tf.ToTensor(),
                                  download='True')

test_set = datasets.FashionMNIST(root='FashionMNIST-data',
                                 train=False,
                                 transform=tf.ToTensor(),
                                 download=True)

print(train_set.classes)

100%|██████████| 26.4M/26.4M [00:05<00:00, 4.72MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 118kB/s]
100%|██████████| 4.42M/4.42M [00:05<00:00, 738kB/s] 
100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]

['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [4]:
# MNIST data, ~ 1024
train_dataloader = DataLoader(train_set, 
                              batch_size=1024,
                              shuffle=True)

test_dataloader = DataLoader(test_set,
                             batch_size=1024)

train_dataloader.dataset

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: FashionMNIST-data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [5]:
# basic custom: 784:128:64:10
class customNN(nn.Module):
    def __init__(self):
        super(customNN, self).__init__()
        
        self.flatten = nn.Flatten()
        self.MLP = nn.Sequential(
            nn.Linear(784, 128),         # 28 x 28
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)           # 10 classes
        )
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.MLP(x)
        return x

model = customNN().to(device)
print(model)

customNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (MLP): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.01)

In [7]:
def training(dataloader, model, loss_fn, optimizer):
    """training field"""
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # forward
        X, y = X.to(device), y.to(device)
        y_hat = model(X)
        loss = loss_fn(y_hat, y)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # for debug
        if batch % 100 == 0:
            loss_print = loss.item()
            print(f'loss: {loss_print}')



def testing_and_metric(dataloader_obj, model, loss_fn):
    """calculate test loss and accuracy metric"""
    num_of_batch = len(dataloader_obj)
    num_of_pic = len(dataloader_obj.dataset)
    loss_test_sum = 0                                                            # calculate mean    
    acc_sum = 0                                                                  # calculate accuracy

    with torch.no_grad():
        for batch, (X, y) in enumerate(dataloader_obj):
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            loss_test_sum += loss.item()                                         # for loss value
            
            acc_sum += (y == pred.argmax(1)).type(torch.float).sum().item()      # class -> dim=1

    model.eval()
    loss_test_mean = loss_test_sum/num_of_batch
    acc_mean = acc_sum/num_of_pic
    print(f"Test Error:\n Accuracy: {100*acc_mean:>.2f}%,      Loss: {loss_test_mean:>.5f} \n")




In [8]:
epoch = 50
for i in range(epoch):
    print(f"Epoch {i+1}\n---------------------")
    training(train_dataloader, model, criterion, optimizer)
    testing_and_metric(test_dataloader, model, criterion)
print('Done!')

Epoch 1
---------------------
loss: 2.311354398727417
Test Error:
 Accuracy: 75.82%,      Loss: 0.66617 

Epoch 2
---------------------
loss: 0.6394752264022827
Test Error:
 Accuracy: 76.80%,      Loss: 0.62361 

Epoch 3
---------------------
loss: 0.5627065300941467
Test Error:
 Accuracy: 79.77%,      Loss: 0.56190 

Epoch 4
---------------------
loss: 0.49286526441574097
Test Error:
 Accuracy: 81.67%,      Loss: 0.49097 

Epoch 5
---------------------
loss: 0.4845823645591736
Test Error:
 Accuracy: 81.24%,      Loss: 0.50900 

Epoch 6
---------------------
loss: 0.47659704089164734
Test Error:
 Accuracy: 82.50%,      Loss: 0.46351 

Epoch 7
---------------------
loss: 0.3906625211238861
Test Error:
 Accuracy: 76.75%,      Loss: 0.67586 

Epoch 8
---------------------
loss: 0.6618348360061646
Test Error:
 Accuracy: 84.46%,      Loss: 0.42083 

Epoch 9
---------------------
loss: 0.36475110054016113
Test Error:
 Accuracy: 82.17%,      Loss: 0.48711 

Epoch 10
---------------------
loss